# 09 · A second curvature factor: the dynamic Svensson model

Three Nelson–Siegel factors with λ = 0.0609 cannot match the published
curves at the short end and at 10 years. Notebook 05 found the fitting
errors there persistent (README §7.6, failure 1), and fixed them after the
fact with AR(1) errors. This notebook fixes the fit instead (README §8,
step 7b, item 8). The three source curves are themselves Svensson or spline
curves, and Svensson (1994) adds a second curvature term with its own decay
λ₂. The dynamic version (De Pooter 2007) makes it a fourth factor in the
state.

**Fixed before the run** (commit `0f0bb3a`):

- **Model.** A 4-factor model estimated exactly like the 3-factor one: EM +
  exact score, 1 bp floor, μ at the real-time sample mean, 6 starts.
- **λ₂ is set by where the second hump peaks** (λτ = 1.7933 at the peak):
  - short: 9 months, λ₂ = 0.1993;
  - long: 90 months, λ₂ = 0.0199.
- **Variants.** Yields only, each plain and with AR(1) errors.

The 3-factor fits are unchanged: a refit reproduces the cached
log-likelihood exactly.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

from yck.eval.backtest import dns_kalman_forecasts, dns_two_step_forecasts, random_walk_forecasts, forward_forecasts
from yck.eval.grid import (STANDARD_GRID, BACKTEST_DEFAULTS, accuracy_grid, anchor_grid, shadow_grid, rwprior_grid,
                           svensson_grid, SVENSSON_LAM2, plan_runs,
                           combination_forecasts, ensure_results, tag_of, _grid_for)
from yck.eval.stats import dm_table, mcs_table

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS, CONFIG, DATA = ROOT / "data/results", ROOT / "config/data_sources.yaml", ROOT / "data"
CCYS = ["USD", "EUR", "GBP"]
H = (1, 6, 12)
SHOW = {"USD": ["3m", "12m", "24m", "60m", "120m"], "EUR": ["3m", "12m", "24m", "60m", "120m"],
        "GBP": ["6m", "12m", "24m", "60m", "120m"]}
FIRST_ORIGIN = {c: BACKTEST_DEFAULTS[c][1] for c in CCYS}
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
CAP = [{"selector": "caption", "props": "font-weight: bold; text-align: left"}]

GRIDS = (STANDARD_GRID, accuracy_grid, anchor_grid, shadow_grid, rwprior_grid, svensson_grid)
for g in GRIDS:  # one grid run if anything is missing
    ensure_results(CCYS, DATA, CONFIG, grid=g)

def name_of(row):
    m, lam, tau, ep, err, anc, sr, rw, l2 = tuple(row) + (None,) * (9 - len(row))
    base = "shadow" if sr else ("policy rate" if m == ("policy_rate",) else "full macro" if m else "yields")
    base += "" if lam == "fixed" else ", λ est."
    base += f", shrink {tau:g}" if tau is not None else ""
    base += f", RW prior {rw:g}" if rw is not None else ""
    base += f", Svensson {'short' if l2 > 0.1 else 'long'}" if l2 is not None else ""
    base += f", endpoint {ep}" if ep else ""
    base += " + AR(1)" if err == "ar1" else " + AR(1) with mean" if err == "ar1c" else ""
    return base + (", anchored" if anc else "") + (f" ({sr})" if sr else "")

y, fc = {}, {}
for c in CCYS:
    est = BACKTEST_DEFAULTS[c][0]
    full = pd.read_parquet(ROOT / f"data/processed/curves_{c}.parquet")
    y[c] = full.loc[est:] if est else full
    fc[c] = {}
    for g in GRIDS:
        for r in _grid_for(g, c):
            fc[c][name_of(r)] = {h: pd.read_parquet(RESULTS / f"fc_{c}_kalman_{tag_of(r)}_h{h}.parquet") for h in H}
    fc[c]["two-step AR(1)"] = dns_two_step_forecasts(y[c], H, FIRST_ORIGIN[c])
    fc[c]["two-step VAR(1)"] = dns_two_step_forecasts(y[c], H, FIRST_ORIGIN[c], kind="var1")
    fc[c]["random walk"] = random_walk_forecasts(y[c], H, FIRST_ORIGIN[c])
    fc[c]["forwards"] = forward_forecasts(pd.read_parquet(ROOT / f"data/processed/curves_ext_{c}.parquet"),
                                          y[c].columns, H, FIRST_ORIGIN[c])
    fc[c].update(combination_forecasts(c, DATA, CONFIG))
SV = [name_of(r) for r in _grid_for(svensson_grid, "USD")]
BASE_OF = {p: "yields" + (" + AR(1)" if "AR(1)" in p else "") for p in SV}
pd.DataFrame({c: {"models compared": len(fc[c])} for c in CCYS})

,USD,EUR,GBP
models compared,64,58,52


## 1. The fit

Measurement-noise sd by maturity at the last refit, and the log-likelihood
gain over the 3-factor model at the first and last refit. The short hump
takes out the short-end misfit and the long hump the 10-year one.

In [2]:
from yck.eval.backtest import dns_kalman_forecasts
rows = {}
for c in CCYS:
    def fits_of(tag, grid):
        kw = [k for cc, t, k in plan_runs([c], DATA, CONFIG, grid=grid) if t == tag][0]
        f = []
        dns_kalman_forecasts(**kw, horizons=(1,), cache_dir=RESULTS / "cache", fits=f)
        return f
    f3 = fits_of("lamfixed", STANDARD_GRID)
    for name, l2 in SVENSSON_LAM2.items():
        f4 = fits_of(f"sv{l2:.4f}_lamfixed", svensson_grid)
        sd3, sd4 = 100 * np.sqrt(f3[-1][1].params.h), 100 * np.sqrt(f4[-1][1].params.h)
        cols = list(f3[-1][1].yield_columns)
        lab = {SHOW[c][0]: "short end (3m; GBP 6m)", "60m": "5y", "120m": "10y"}
        rows[(c, name)] = {**{f"noise sd {lab[m]}, 3f → 4f (bp)": f"{sd3[cols.index(m)]:.1f} → {sd4[cols.index(m)]:.1f}"
                              for m in lab},
                           "log-lik gain, first / last refit": f"{f4[0][1].loglik - f3[0][1].loglik:.0f} / "
                                                               f"{f4[-1][1].loglik - f3[-1][1].loglik:.0f}"}
pd.DataFrame(rows).T

noise sd short end (3m; GBP 6m), 3f → 4f (bp)  \
USD short                                    19.3 → 5.9   
    long                                    19.3 → 17.5   
EUR short                                    19.9 → 7.5   
    long                                    19.9 → 17.1   
GBP short                                    16.0 → 1.0   
    long                                    16.0 → 13.2   

          noise sd 5y, 3f → 4f (bp) noise sd 10y, 3f → 4f (bp)  \
USD short                 6.0 → 5.3                14.1 → 13.5   
    long                  6.0 → 1.0                 14.1 → 2.6   
EUR short                 6.7 → 6.0                14.9 → 14.2   
    long                  6.7 → 1.0                 14.9 → 2.8   
GBP short                 3.8 → 3.8                12.6 → 12.7   
    long                  3.8 → 1.0                 12.6 → 1.3   

          log-lik gain, first / last refit  
USD short                       860 / 2144  
    long                       1850 / 5801  
EUR short                       249 / 1275  
    long                        710 / 3236  
GBP short                       886 / 1658  
    long                       1367 / 4182

## 2. Against the 3-factor model

RMSE ratio, Svensson ÷ 3-factor model with the same error model
(Diebold–Mariano, two-sided stars; < 1 means the fourth factor helps).

In [3]:
def after(c, h):
    return y[c].loc[pd.Timestamp(FIRST_ORIGIN[c]) + pd.offsets.MonthEnd(h):]

def vs(c, bench, h, names, field="cell"):
    t = dm_table({n: fc[c][n][h] for n in [bench] + names}, bench, after(c, h), h, SHOW[c])
    return t.xs(field, axis=1, level=1).T

for c in CCYS:
    t = pd.concat({f"h = {h}": pd.DataFrame({p: vs(c, BASE_OF[p], h, [p]).loc[p] for p in SV}).T for h in H}, axis=1)
    display(t.style.set_caption(f"{c}: Svensson ÷ 3-factor").set_table_styles(CAP))

## 3. Against forwards and the random walk

In [4]:
SHOWN = ["yields", "yields + AR(1)"] + SV + ["combination A, inverse MSE"]
for c in CCYS:
    t = pd.concat({**{f"÷ forwards, h = {h}": vs(c, "forwards", h, SHOWN) for h in H},
                   **{f"÷ RW, h = {h}": vs(c, "random walk", h, SHOWN, "ratio").astype(float).round(2) for h in H}},
                  axis=1)
    display(t.style.set_caption(f"{c}").set_table_styles(CAP))

## 4. Model confidence set over everything

The 90% MCS over every model in notebooks 02–09 (pooled squared-error
loss).

In [5]:
SHORT = "3m / 6m (GBP)"
mcs = {(c, h): mcs_table({n: f[h] for n, f in fc[c].items()}, after(c, h), h, SHOW[c], n_boot=2000)
               .rename(columns={SHOW[c][0]: SHORT})
       for c in CCYS for h in H}
summary = {(c, f"h = {h}"): {col: f"{int((t[col] >= 0.10).sum())} / {int(t[col].notna().sum())}" for col in t.columns}
           for (c, h), t in mcs.items()}
display(pd.DataFrame(summary).T.style.set_caption("Models in the 90% MCS / models compared").set_table_styles(CAP))
KEY = ["yields", "yields + AR(1)"] + SV + ["combination A, inverse MSE", "random walk", "forwards"]
pd.DataFrame({(c, f"h = {h}"): mcs[(c, h)].loc[KEY, "all"] for c in CCYS for h in H}).round(2)

USD                EUR                GBP  \
                               h = 1 h = 6 h = 12 h = 1 h = 6 h = 12 h = 1   
model                                                                        
yields                          0.00  0.95    1.0  0.00  0.40   0.55  0.00   
yields + AR(1)                  0.82  0.98    1.0  0.08  0.35   0.49  0.84   
yields, Svensson short          0.03  0.99    1.0  0.20  0.56   0.44  0.03   
yields, Svensson short + AR(1)  0.98  0.99    1.0  0.20  0.56   0.42  0.87   
yields, Svensson long           0.73  0.99    1.0  0.07  0.40   0.37  0.84   
yields, Svensson long + AR(1)   0.98  0.99    1.0  0.20  0.40   0.39  0.84   
combination A, inverse MSE      1.00  0.99    1.0  0.87  0.96   0.86  0.84   
random walk                     0.63  0.99    1.0  0.87  0.96   0.95  0.87   
forwards                        1.00  0.99    1.0  1.00  1.00   1.00   NaN   

                                             
                               h = 6 h = 12  
model                                        
yields                          0.76   0.89  
yields + AR(1)                  0.76   0.89  
yields, Svensson short          1.00   0.92  
yields, Svensson short + AR(1)  1.00   0.92  
yields, Svensson long           0.76   0.89  
yields, Svensson long + AR(1)   0.76   0.89  
combination A, inverse MSE      0.97   0.99  
random walk                     1.00   0.99  
forwards                        0.93   0.89

## 5. Takeaways

1. **The fourth factor removes the misfit it was aimed at.** Noise sd at
   the last refit:

   | | Short hump, short end | Long hump, 10y |
   |---|---|---|
   | USD | 19.3 → 5.9 bp (3m) | 14.1 → 2.6 bp |
   | EUR | 19.9 → 7.5 bp (3m) | 14.9 → 2.8 bp |
   | GBP | 16.0 → 1.0 bp (6m) | 12.6 → 1.3 bp |

   The log-likelihood gains are 1,275–5,801 points.
2. **The short hump is the one that forecasts better.** Against the
   3-factor model at h = 1:

   | | Plain | With AR(1) on both sides |
   |---|---|---|
   | USD 3m | −13% (1%) | −3% |
   | EUR 3m | −48% (1%) | −32% (1%) |
   | GBP 6m | −14% (5%) | −4% |

   - **Against forwards at one month,** GBP's 6m goes from 3.29 (1.89 with
     AR(1) errors) to 1.06 (not significant): the first model level with
     forwards at the GBP short end.
   - **h = 6:**
     - GBP 6m −9 to −14% (significant at 5–10%);
     - EUR 3m −19 to −20% (10%);
     - USD 1–4% better (not significant).
   - **h = 12:**
     - USD 5y and 10y −4 to −5% (10y at 0.94–0.95 × forwards);
     - GBP 6m–2y at 0.92–0.95 × forwards (not significant);
     - EUR 1–2y 9–13% *worse*.
   - **Confidence set.** At h = 6 in GBP, the short-hump model has
     p = 1.00, the best of all 52 GBP models on the pooled loss.
3. **The long hump only repeats what AR(1) errors already do.** It cuts
   the one-month 10y error (USD −10%, GBP −16%; EUR +6%), but with AR(1)
   errors on both sides the gain is gone. At 6–12 months it adds nothing,
   and it hurts the EUR 10y (+11–16%). The 10y misfit is persistent but
   carries no information about where the 10y yield is going.
4. **Still nothing beats the random walk or forwards significantly at
   6–12 months in USD or EUR,** and the confidence set keeps 55–64 of
   58–64 models there.
5. **What carries forward.**
   - **USD and GBP:** the short-hump Svensson model + AR(1) becomes the
     pure-model signal.
   - **EUR:** the shadow-rate model (notebook 07) stays.
   - **Combination A** is unchanged.